# NSGA-II多目标优化算法

## 什么是多目标优化？

多目标优化问题涉及同时优化多个目标函数。与单目标优化不同，多目标优化通常不存在单一的最优解，而是一个解集，称为帕累托前沿（Pareto Front）。

## NSGA-II算法简介

NSGA-II（非支配排序遗传算法II）是经典的多目标优化算法，特点：
- 快速非支配排序
- 拥挤度计算保持多样性
- 精英保留策略

## 偏置驱动的多目标优化

在NSGABLACK框架中，**偏置系统**为多目标优化提供了强大的增强能力：

### 1. 多目标偏置类型
- **Pareto引导偏置**：向稀疏的Pareto区域引导搜索
- **偏好导向偏置**：基于决策者偏好调整搜索方向
- **参考点偏置**：向参考点区域集中搜索
- **多样性偏置**：保持解集分布均匀

### 2. 业务场景示例
- **投资组合优化**：平衡收益与风险
- **产品设计**：平衡成本、性能与可靠性
- **供应链管理**：平衡成本、时间与服务质量
- **能源调度**：平衡成本、排放与可靠性

### 3. 偏置在多目标优化中的独特价值
- **引导收敛**：加速向Pareto前沿收敛
- **改善分布**：获得更均匀的解集分布
- **融入偏好**：反映决策者真实需求
- **处理约束**：智能避开不可行区域

In [ ]:
# 导入必要的库
import numpy as np
import matplotlib.pyplot as plt
import random
from typing import List, Tuple, Dict, Any, Optional
from abc import ABC, abstractmethod
from collections import defaultdict

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

print("环境准备完成")
print(f"NumPy 版本: {np.__version__}")

## 多目标偏置系统定义

In [ ]:
class MultiObjectiveBias(ABC):
    """多目标偏置基类"""
    
    def __init__(self, name: str, strength: float = 1.0):
        self.name = name
        self.strength = strength
        self.enabled = True
        self.history = []
        
    @abstractmethod
    def apply_bias(self, population: List['Individual'], 
                    context: Dict[str, Any] = None) -> List['Individual']:
        """应用偏置到种群
        
        Args:
            population: 当前种群
            context: 上下文信息（Pareto前沿、参考点等）
            
        Returns:
            偏置后的种群
        """
        pass
    
    def update_context(self, pareto_front: List['Individual'] = None):
        """更新偏置上下文"""
        if pareto_front:
            self.history.append(len(pareto_front))


class ParetoGuidanceBias(MultiObjectiveBias):
    """Pareto引导偏置 - 向稀疏区域引导搜索"""
    
    def __init__(self, strength: float = 1.0, density_threshold: float = 0.1):
        super().__init__("Pareto引导偏置", strength)
        self.density_threshold = density_threshold
        
    def apply_bias(self, population, context=None):
        if not context or 'pareto_front' not in context:
            return population
        
        pareto_front = context['pareto_front']
        if len(pareto_front) < 3:
            return population
        
        biased_population = []
        
        for ind in population:
            # 计算个体在Pareto前沿附近的密度
            density = self._calculate_local_density(ind.objectives, pareto_front)
            
            if density < self.density_threshold:
                # 低密度区域，应用吸引偏置
                bias_direction = self._calculate_bias_direction(ind.objectives, pareto_front)
                
                # 调整决策变量
                biased_decision = self._apply_decision_bias(ind.decision, bias_direction)
                
                new_ind = Individual(len(ind.decision))
                new_ind.decision = biased_decision
                biased_population.append(new_ind)
            else:
                # 高密度区域，保持原样或轻微扰动
                biased_population.append(ind)
        
        return biased_population
    
    def _calculate_local_density(self, objectives, pareto_front):
        """计算局部密度"""
        distances = []
        for p_ind in pareto_front:
            dist = np.sqrt(sum((obj_i - p_obj_i)**2 
                             for obj_i, p_obj_i in zip(objectives, p_ind.objectives)))
            distances.append(dist)
        
        # 使用k近邻距离作为密度指标
        k = min(5, len(distances))
        k_nearest = sorted(distances)[:k]
        return sum(k_nearest) / k
    
    def _calculate_bias_direction(self, objectives, pareto_front):
        """计算偏置方向"""
        # 找到最近的稀疏区域
        min_density = float('inf')
        target_point = None
        
        for p_ind in pareto_front:
            density = self._calculate_local_density(p_ind.objectives, pareto_front)
            if density < min_density:
                min_density = density
                target_point = p_ind.objectives
        
        if target_point:
            direction = np.array(target_point) - np.array(objectives)
            if np.linalg.norm(direction) > 0:
                return direction / np.linalg.norm(direction)
        
        return np.zeros(len(objectives))
    
    def _apply_decision_bias(self, decision, bias_direction):
        """在决策空间应用偏置（简化版）"""
        # 将目标空间偏置转换为决策空间偏置
        # 简化实现：添加随机扰动
        noise = np.random.randn(len(decision)) * 0.1 * self.strength
        
        biased_decision = decision + noise
        
        # 确保在[0,1]范围内
        return np.clip(biased_decision, 0, 1)


class PreferenceGuidedBias(MultiObjectiveBias):
    """偏好导向偏置 - 基于决策者偏好引导搜索"""
    
    def __init__(self, strength: float = 1.0, reference_point=None, 
                 preference_weights=None):
        super().__init__("偏好导向偏置", strength)
        self.reference_point = reference_point or [0.5, 0.5]
        self.preference_weights = preference_weights or [0.5, 0.5]
        
    def apply_bias(self, population, context=None):
        biased_population = []
        
        for ind in population:
            # 计算偏好适应度
            preference_fitness = self._calculate_preference_fitness(ind.objectives)
            
            # 基于偏好适应度调整搜索方向
            if preference_fitness < 0.5:  # 远离偏好
                # 向参考点方向偏置
                bias_strength = self.strength * (0.5 - preference_fitness)
                biased_decision = self._bias_toward_reference(ind.decision, bias_strength)
            else:
                # 保持搜索多样性
                biased_decision = self._maintain_diversity(ind.decision)
            
            new_ind = Individual(len(ind.decision))
            new_ind.decision = biased_decision
            biased_population.append(new_ind)
        
        return biased_population
    
    def _calculate_preference_fitness(self, objectives):
        """计算偏好适应度"""
        # 加权距离参考点
        weighted_distance = sum(w * abs(obj - ref) 
                                 for w, obj, ref in zip(self.preference_weights, 
                                                    objectives, 
                                                    self.reference_point))
        
        # 归一化到[0,1]
        return 1 / (1 + weighted_distance)
    
    def _bias_toward_reference(self, decision, strength):
        """向参考点方向偏置决策变量"""
        # 简化实现：随机调整
        adjustment = np.random.randn(len(decision)) * strength
        return np.clip(decision + adjustment, 0, 1)
    
    def _maintain_diversity(self, decision):
        """保持多样性"""
        # 添加小的随机扰动
        noise = np.random.randn(len(decision)) * 0.05
        return np.clip(decision + noise, 0, 1)


class DiversityPreservationBias(MultiObjectiveBias):
    """多样性保持偏置 - 确保解集均匀分布"""
    
    def __init__(self, strength: float = 1.0, spacing_threshold: float = 0.05):
        super().__init__("多样性保持偏置", strength)
        self.spacing_threshold = spacing_threshold
        self.archive = []  # 维护解的档案
        
    def apply_bias(self, population, context=None):
        # 更新档案
        if context and 'pareto_front' in context:
            self._update_archive(context['pareto_front'])
        
        biased_population = []
        
        for ind in population:
            # 检查是否与档案中的解过于接近
            too_close = False
            for archived_ind in self.archive:
                if self._are_too_close(ind.objectives, archived_ind.objectives):
                    too_close = True
                    break
            
            if too_close:
                # 应用排斥偏置
                biased_decision = self._apply_repulsion_bias(ind.decision)
            else:
                biased_decision = ind.decision
            
            new_ind = Individual(len(ind.decision))
            new_ind.decision = biased_decision
            biased_population.append(new_ind)
        
        return biased_population
    
    def _update_archive(self, pareto_front):
        """更新档案"""
        # 添加新解到档案
        for ind in pareto_front:
            is_dominated = False
            for archived_ind in self.archive:
                if self._dominates(archived_ind.objectives, ind.objectives):
                    is_dominated = True
                    break
            
            if not is_dominated:
                # 移除被新解支配的档案解
                self.archive = [a for a in self.archive 
                                if not self._dominates(ind.objectives, a.objectives)]
                self.archive.append(ind)
        
        # 限制档案大小
        if len(self.archive) > 100:
            # 保持多样性，移除拥挤的解
            self.archive = self._select_diverse_subset(self.archive, 50)
    
    def _are_too_close(self, obj1, obj2, threshold=None):
        """检查两个目标向量是否过于接近"""
        if threshold is None:
            threshold = self.spacing_threshold
        
        distance = np.sqrt(sum((o1 - o2)**2 for o1, o2 in zip(obj1, obj2)))
        return distance < threshold
    
    def _dominates(self, obj1, obj2):
        """检查obj1是否支配obj2"""
        better_in_all = all(o1 <= o2 for o1, o2 in zip(obj1, obj2))
        better_in_at_least_one = any(o1 < o2 for o1, o2 in zip(obj1, obj2))
        return better_in_all and better_in_at_least_one
    
    def _apply_repulsion_bias(self, decision):
        """应用排斥偏置"""
        # 随机扰动避免聚集
        perturbation = np.random.randn(len(decision)) * self.strength * 0.2
        return np.clip(decision + perturbation, 0, 1)
    
    def _select_diverse_subset(self, individuals, size):
        """选择多样性子集"""
        if len(individuals) <= size:
            return individuals
        
        selected = [individuals[0]]
        
        while len(selected) < size:
            # 选择与现有选择距离最远的个体
            best_idx = 0
            max_min_distance = -1
            
            for i, ind in enumerate(individuals):
                if ind in selected:
                    continue
                
                min_distance = min(self._calculate_distance(ind.objectives, sel.objectives) 
                                for sel in selected)
                
                if min_distance > max_min_distance:
                    max_min_distance = min_distance
                    best_idx = i
            
            selected.append(individuals[best_idx])
        
        return selected
    
    def _calculate_distance(self, obj1, obj2):
        """计算目标空间距离"""
        return np.sqrt(sum((o1 - o2)**2 for o1, o2 in zip(obj1, obj2)))


print("多目标偏置系统已定义")

## 定义多目标测试问题和个体类

In [ ]:
# 重新定义Individual类（适应偏置系统）
class Individual:
    """个体类"""
    
    def __init__(self, n_var):
        self.decision = np.random.uniform(0, 1, n_var)
        self.objectives = None
        self.rank = 0
        self.crowding_distance = 0
        self.dominated_count = 0
        self.dominates = []
        self.constraint_violation = 0
        
    def dominates_other(self, other):
        """判断是否支配另一个个体"""
        # 检查约束
        if self.constraint_violation > 0 and other.constraint_violation > 0:
            # 都不可行，违反程度小的更好
            return self.constraint_violation < other.constraint_violation
        elif self.constraint_violation == 0 and other.constraint_violation > 0:
            # 可行优于不可行
            return True
        elif self.constraint_violation > 0 and other.constraint_violation == 0:
            # 不可行劣于可行
            return False
        else:
            # 都可行，检查目标支配关系
            at_least_one_better = False
            
            for i in range(len(self.objectives)):
                if self.objectives[i] > other.objectives[i]:
                    return False
                if self.objectives[i] < other.objectives[i]:
                    at_least_one_better = True
            
            return at_least_one_better


# 定义多目标测试问题
class ZDT1:
    """ZDT1测试问题 - 经典的2目标测试问题"""
    
    def __init__(self, n_var=30):
        self.n_var = n_var
        self.n_obj = 2
        
    def evaluate(self, x):
        """评估解x
        
        Args:
            x: 决策变量列表
        
        Returns:
            目标值列表 [f1, f2]
        """
        f1 = x[0]
        
        g = 1 + 9 * sum(x[1:]) / (self.n_var - 1)
        
        h = 1 - np.sqrt(f1 / g)
        
        f2 = g * h
        
        return [f1, f2]


class ConstrainedZDT1:
    """带约束的ZDT1问题 - 业务场景模拟"""
    
    def __init__(self, n_var=10, budget_limit=5.0):
        self.n_var = n_var
        self.n_obj = 2
        self.budget_limit = budget_limit
        
    def evaluate(self, x):
        """评估解x，包含目标函数和约束"""
        # 目标函数
        f1 = x[0]
        
        g = 1 + 9 * sum(x[1:]) / (self.n_var - 1)
        
        h = 1 - np.sqrt(f1 / g)
        
        f2 = g * h
        
        # 约束：资源约束
        resource_usage = sum(x) * 10  # 资源使用量
        
        # 约束：质量约束
        quality = 1 - f1  # 质量分数
        
        return [f1, f2]
    
    def evaluate_constraints(self, x):
        """评估约束违反程度"""
        violations = []
        
        # 预算约束
        budget_usage = sum(x) * 10
        if budget_usage > self.budget_limit:
            violations.append(budget_usage - self.budget_limit)
        
        # 质量约束（隐式在目标中）
        if x[0] > 0.8:  # 第一个目标不能太高
            violations.append(x[0] - 0.8)
        
        # 决策变量边界约束
        for i, val in enumerate(x):
            if i < 3:  # 前3个变量有特殊约束
                if val > 0.7:
                    violations.append(val - 0.7)
        
        return sum(violations)


# 创建测试问题
print("创建多目标测试问题：")
print("-"*40)

# 标准测试问题
zdt1 = ZDT1(n_var=10)
print(f"ZDT1: {zdt1.n_var}个变量，{zdt1.n_obj}个目标")

# 业务场景：投资组合优化
investment_problem = ConstrainedZDT1(n_var=10, budget_limit=5.0)
print(f"投资组合问题: {investment_problem.n_var}个投资项目，预算限制{investment_problem.budget_limit}")

# 测试评估函数
test_x = [0.5] + [0.5] * 9
print(f"\n测试解: {test_x[:3]}...")
print(f"ZDT1目标值: {zdt1.evaluate(test_x)}")
print(f"投资组合约束违反: {investment_problem.evaluate_constraints(test_x)}")

## 偏置引导的NSGA-II算法

In [ ]:
class BiasedNSGA2:
    """偏置引导的NSGA-II算法"""
    
    def __init__(self, problem, pop_size=100, max_gen=100, 
                 crossover_rate=0.9, mutation_rate=1.0/pop_size[0] if isinstance(pop_size, tuple) else 1.0/100,
                 biases: List[MultiObjectiveBias] = None):
        self.problem = problem
        self.pop_size = pop_size[0] if isinstance(pop_size, tuple) else pop_size
        self.max_gen = max_gen
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate
        self.biases = biases or []
        
        self.population = []
        self.history = []
        self.bias_statistics = {bias.name: 0 for bias in self.biases}
        
        print(f"创建偏置引导NSGA-II算法")
        print(f"  种群大小: {self.pop_size}")
        print(f"  最大代数: {max_gen}")
        print(f"  交叉率: {crossover_rate}")
        print(f"  变异率: {mutation_rate}")
        print(f"  激活偏置: {[bias.name for bias in self.biases]}")
    
    def initialize_population(self):
        """初始化种群"""
        self.population = []
        for _ in range(self.pop_size):
            individual = Individual(self.problem.n_var)
            individual.objectives = self.problem.evaluate(individual.decision)
            
            # 评估约束（如果有）
            if hasattr(self.problem, 'evaluate_constraints'):
                individual.constraint_violation = self.problem.evaluate_constraints(individual.decision)
            
            self.population.append(individual)
    
    def apply_biases(self, population, generation):
        """应用偏置到种群"""
        if not self.biases:
            return population
        
        # 构建上下文信息
        context = {
            'generation': generation,
            'pareto_front': None
        }
        
        # 获取当前Pareto前沿
        if self.population:
            fronts = self.fast_non_dominated_sort()
            if fronts:
                context['pareto_front'] = fronts[0]
        
        # 应用偏置
        biased_population = population
        for bias in self.biases:
            if bias.enabled:
                pre_size = len(biased_population)
                biased_population = bias.apply_bias(biased_population, context)
                
                # 记录偏置效果
                if len(biased_population) != pre_size:
                    self.bias_statistics[bias.name] += 1
                
                # 更新偏置上下文
                bias.update_context(context['pareto_front'])
        
        return biased_population
    
    def fast_non_dominated_sort(self):
        """快速非支配排序"""
        fronts = [[]]
        
        # 计算支配关系
        for ind in self.population:
            ind.dominated_count = 0
            ind.dominates = []
            
            for other in self.population:
                if ind.dominates_other(other):
                    ind.dominates.append(other)
                elif other.dominates_other(ind):
                    ind.dominated_count += 1
            
            if ind.dominated_count == 0:
                ind.rank = 0
                fronts[0].append(ind)
        
        # 创建其他前沿
        i = 0
        while fronts[i]:
            next_front = []
            
            for ind in fronts[i]:
                for other in ind.dominates:
                    other.dominated_count -= 1
                    if other.dominated_count == 0:
                        other.rank = i + 1
                        next_front.append(other)
            
            i += 1
            fronts.append(next_front)
        
        return fronts[:-1]  # 移除最后一个空前沿
    
    def calculate_crowding_distance(self, front):
        """计算拥挤度"""
        if len(front) == 0:
            return
        
        n_obj = len(front[0].objectives)
        n_ind = len(front)
        
        # 初始化拥挤度
        for ind in front:
            ind.crowding_distance = 0
        
        # 为每个目标计算拥挤度
        for m in range(n_obj):
            # 按第m个目标排序
            front.sort(key=lambda x: x.objectives[m])
            
            # 边界点设为无穷大
            front[0].crowding_distance = float('inf')
            front[-1].crowding_distance = float('inf')
            
            # 计算中间点的拥挤度
            obj_min = front[0].objectives[m]
            obj_max = front[-1].objectives[m]
            
            if obj_max - obj_min > 1e-10:
                for i in range(1, n_ind - 1):
                    distance = front[i+1].objectives[m] - front[i-1].objectives[m]
                    front[i].crowding_distance += distance / (obj_max - obj_min)
    
    def tournament_selection(self, population, tournament_size=2):
        """锦标赛选择"""
        selected = []
        
        while len(selected) < len(population):
            # 随机选择个体进行锦标赛
            candidates = random.sample(population, min(tournament_size, len(population)))
            
            # 选择最好的（非支配排序等级小，拥挤度大）
            best = candidates[0]
            for candidate in candidates[1:]:
                if (candidate.rank < best.rank or 
                    (candidate.rank == best.rank and candidate.crowding_distance > best.crowding_distance)):
                    best = candidate
            
            selected.append(best)
        
        return selected
    
    def sbx_crossover(self, parent1, parent2):
        """模拟二进制交叉（SBX）"""
        if random.random() > self.crossover_rate:
            return parent1.decision.copy(), parent2.decision.copy()
        
        child1 = parent1.decision.copy()
        child2 = parent2.decision.copy()
        
        eta_c = 15  # 交叉分布指数
        
        for i in range(len(child1)):
            if random.random() <= 0.5:
                y1 = min(child1[i], child2[i])
                y2 = max(child1[i], child2[i])
                
                if abs(y1 - y2) > 1e-10:
                    rand = random.random()
                    beta = 1.0 / (2.0 - rand)
                    
                    alpha = 2.0 - pow(beta, -(eta_c + 1.0))
                    
                    if rand <= 1.0 / alpha:
                        beta_q = pow(rand * alpha, 1.0 / (eta_c + 1.0))
                    else:
                        beta_q = pow(1.0 / (2.0 - rand * alpha), 1.0 / (eta_c + 1.0))
                    
                    c1 = 0.5 * ((y1 + y2) - beta_q * (y2 - y1))
                    c2 = 0.5 * ((y1 + y2) + beta_q * (y2 - y1))
                    
                    c1 = max(0, min(1, c1))
                    c2 = max(0, min(1, c2))
                    
                    if random.random() <= 0.5:
                        child1[i] = c2
                        child2[i] = c1
                    else:
                        child1[i] = c1
                        child2[i] = c2
        
        return child1, child2
    
    def polynomial_mutation(self, individual):
        """多项式变异"""
        mutated = individual.decision.copy()
        
        eta_m = 20  # 变异分布指数
        
        for i in range(len(mutated)):
            if random.random() <= self.mutation_rate:
                y = mutated[i]
                
                delta_l = y
                delta_u = 1 - y
                
                rand = random.random()
                
                if rand <= 0.5:
                    xy = 1 - delta_l
                    val = 2.0 * rand + (1.0 - 2.0 * rand) * pow(xy, eta_m + 1.0)
                    delta_q = pow(val, 1.0 / (eta_m + 1.0)) - 1.0
                else:
                    xy = 1 - delta_u
                    val = 2.0 * (1.0 - rand) + 2.0 * (rand - 0.5) * pow(xy, eta_m + 1.0)
                    delta_q = 1.0 - pow(val, 1.0 / (eta_m + 1.0))
                
                y = y + delta_q
                y = max(0, min(1, y))
                
                mutated[i] = y
        
        return mutated
    
    def create_offspring(self):
        """创建子代"""
        offspring = []
        
        # 锦标赛选择
        selected = self.tournament_selection(self.population)
        
        # 交叉和变异
        for i in range(0, len(selected), 2):
            if i + 1 < len(selected):
                # 交叉
                child1_dec, child2_dec = self.sbx_crossover(selected[i], selected[i+1])
                
                # 变异
                child1_dec = self.polynomial_mutation(type('obj', (object,), {'decision': child1_dec})())
                child2_dec = self.polynomial_mutation(type('obj', (object,), {'decision': child2_dec})())
                
                # 创建新个体
                child1 = Individual(self.problem.n_var)
                child1.decision = np.array(child1_dec)
                child1.objectives = self.problem.evaluate(child1.decision)
                
                if hasattr(self.problem, 'evaluate_constraints'):
                    child1.constraint_violation = self.problem.evaluate_constraints(child1.decision)
                
                child2 = Individual(self.problem.n_var)
                child2.decision = np.array(child2_dec)
                child2.objectives = self.problem.evaluate(child2.decision)
                
                if hasattr(self.problem, 'evaluate_constraints'):
                    child2.constraint_violation = self.problem.evaluate_constraints(child2.decision)
                
                offspring.extend([child1, child2])
        
        return offspring
    
    def environmental_selection(self):
        """环境选择"""
        # 合并父代和子代
        combined = self.population + self.offspring
        
        # 快速非支配排序
        fronts = self.fast_non_dominated_sort()
        
        # 计算拥挤度
        for front in fronts:
            self.calculate_crowding_distance(front)
        
        # 选择新一代
        new_population = []
        
        for front in fronts:
            if len(new_population) + len(front) <= self.pop_size:
                new_population.extend(front)
            else:
                # 按拥挤度排序
                front.sort(key=lambda x: x.crowding_distance, reverse=True)
                remaining = self.pop_size - len(new_population)
                new_population.extend(front[:remaining])
                break
        
        self.population = new_population
    
    def optimize(self):
        """执行优化"""
        print(f"\n开始偏置引导NSGA-II优化...")
        
        # 初始化种群
        self.initialize_population()
        print(f"初始种群已生成，大小: {len(self.population)}")
        
        # 进化循环
        for generation in range(self.max_gen):
            # 应用偏置（从第10代开始）
            if generation >= 10:
                self.population = self.apply_biases(self.population, generation)
            
            # 创建子代
            self.offspring = self.create_offspring()
            
            # 环境选择
            self.environment_selection()
            
            # 记录历史
            front = [ind for ind in self.population if ind.rank == 0]
            self.history.append(front)
            
            # 输出进度
            if (generation + 1) % 20 == 0:
                print(f"  代数: {generation + 1}/{self.max_gen}, "
                      f"第一前沿大小: {len(front)}, "
                      f"活跃偏置: {sum(1 for b in self.biases if b.enabled and generation > 10)}")
        
        print(f"\n优化完成！")
        print(f"最终第一前沿大小: {len([ind for ind in self.population if ind.rank == 0])}")
        
        # 输出偏置统计
        if self.bias_statistics:
            print("\n偏置应用统计：")
            for bias_name, count in self.bias_statistics.items():
                if count > 0:
                    print(f"  {bias_name}: {count}次")
        
        # 返回帕累托前沿
        pareto_front = [ind for ind in self.population if ind.rank == 0]
        return pareto_front


print("偏置引导NSGA-II算法已定义")

## 业务场景：投资组合优化配置

In [ ]:
# 定义业务配置
print("投资组合优化业务配置：")
print("="*50)

investment_config = {
    'assets': [
        {'name': '科技股', 'expected_return': 0.15, 'risk': 0.25, 'min_weight': 0.1, 'max_weight': 0.4},
        {'name': '国债', 'expected_return': 0.05, 'risk': 0.05, 'min_weight': 0.2, 'max_weight': 0.5},
        {'name': '房地产', 'expected_return': 0.10, 'risk': 0.15, 'min_weight': 0.1, 'max_weight': 0.3},
        {'name': '大宗商品', 'expected_return': 0.12, 'risk': 0.20, 'min_weight': 0.05, 'max_weight': 0.25},
        {'name': '现金', 'expected_return': 0.02, 'risk': 0.01, 'min_weight': 0.05, 'max_weight': 0.2}
    ],
    'total_budget': 1000000,  # 总预算
    'risk_tolerance': 0.15,   # 风险容忍度
    'min_diversification': 3,    # 最少资产数量
    'max_single_allocation': 0.35  # 单项资产最大配置比例
}

print("投资目标：")
print("- 最大化收益")
print("- 最小化风险")
print("\n资产配置约束：")
for asset in investment_config['assets']:
    print(f"- {asset['name']}: 收益{asset['expected_return']:.0%}, "
          f"风险{asset['risk']:.0%}, "
          f"配置范围{asset['min_weight']:.0%}-{asset['max_weight']:.0%}")

print(f"\n总体约束：")
print(f"- 总预算: {investment_config['total_budget']:,}")
print(f"- 风险容忍度: {investment_config['risk_tolerance']:.0%}")
print(f"- 最少配置: {investment_config['min_diversification']}种资产")
print(f"- 单项上限: {investment_config['max_single_allocation']:.0%}")

## 对比实验：标准NSGA-II vs 偏置引导NSGA-II

In [ ]:
# 实验设置
max_gen = 100
pop_size = 100
seed = 42
np.random.seed(seed)
random.seed(seed)

print("对比实验：标准NSGA-II vs 偏置引导NSGA-II")
print("="*60)

# 1. 标准NSGA-II（无偏置）
print("\n1. 标准NSGA-II优化：")
print("-"*30)
standard_nsga2 = BiasedNSGA2(investment_problem, pop_size=pop_size, max_gen=max_gen, biases=[])
standard_pareto = standard_nsga2.optimize()

# 2. 偏置引导NSGA-II
print("\n\n2. 偏置引导NSGA-II优化：")
print("-"*30)

# 创建偏置组合
biases = [
    ParetoGuidanceBias(strength=0.3, density_threshold=0.1),
    PreferenceGuidedBias(
        strength=0.4,
        reference_point=[0.1, 0.1],  # 偏好低风险
        preference_weights=[0.3, 0.7]  # 更重视风险控制
    ),
    DiversityPreservationBias(strength=0.3, spacing_threshold=0.05)
]

biased_nsga2 = BiasedNSGA2(
    investment_problem, 
    pop_size=pop_size, 
    max_gen=max_gen, 
    biases=biases
)
biased_pareto = biased_nsga2.optimize()

# 3. 单一偏置效果对比
print("\n\n3. 单一偏置效果对比：")
print("-"*30)

single_bias_results = {}
bias_configs = [
    ("仅Pareto引导", [ParetoGuidanceBias(strength=0.5)]),
    ("仅偏好导向", [PreferenceGuidedBias(strength=0.5, reference_point=[0.1, 0.1])]),
    ("仅多样性保持", [DiversityPreservationBias(strength=0.5)])
]

for config_name, bias_list in bias_configs:
    print(f"\n{config_name}：")
    nsga2_single = BiasedNSGA2(
        investment_problem, 
        pop_size=pop_size//2, 
        max_gen=max_gen//2, 
        biases=bias_list
    )
    pareto = nsga2_single.optimize()
    single_bias_results[config_name] = pareto

# 提取目标值用于可视化
standard_objectives = np.array([ind.objectives for ind in standard_pareto])
biased_objectives = np.array([ind.objectives for ind in biased_pareto])

# 计算约束满足率
def calculate_feasibility_rate(pareto_front):
    feasible_count = sum(1 for ind in pareto_front if ind.constraint_violation <= 0)
    return feasible_count / len(pareto_front)

standard_feasible = calculate_feasibility_rate(standard_pareto)
biased_feasible = calculate_feasibility_rate(biased_pareto)

# 可视化对比
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. 帕累托前沿对比
ax1 = axes[0, 0]
ax1.scatter(standard_objectives[:, 0], standard_objectives[:, 1], 
           alpha=0.6, s=50, label='标准NSGA-II')
ax1.scatter(biased_objectives[:, 0], biased_objectives[:, 1], 
           alpha=0.6, s=50, label='偏置引导NSGA-II')
ax1.set_xlabel('成本 (f1)')
ax1.set_ylabel('风险 (f2)')
ax1.set_title('帕累托前沿对比')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 添加参考线
ax1.axvline(x=0.8, color='red', linestyle='--', alpha=0.5, label='质量约束')
ax1.text(0.81, ax1.get_ylim()[1]*0.9, '约束', color='red', fontsize=9)

# 2. 解的质量指标
ax2 = axes[0, 1]
methods = ['标准NSGA-II', '偏置引导NSGA-II']
sizes = [len(standard_pareto), len(biased_pareto)]
feasible_rates = [standard_feasible, biased_feasible]
colors = ['blue', 'red']

x = np.arange(len(methods))
width = 0.35

bars1 = ax2.bar(x - width/2, sizes, width, label='解的数量', alpha=0.7, color=colors)
bars2 = ax2.bar(x + width/2, [r*100 for r in feasible_rates], width, 
                label='可行解比例(%)', alpha=0.7, color=colors)
ax2.set_ylabel('数量 / 比例')
ax2.set_title('解的质量对比')
ax2.set_xticks(x)
ax2.set_xticklabels(methods)
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# 3. 收敛历史
ax3 = axes[1, 0]
standard_sizes = [len(front) for front in standard_nsga2.history]
biased_sizes = [len(front) for front in biased_nsga2.history]
ax3.plot(standard_sizes, 'b-', label='标准NSGA-II', linewidth=2)
ax3.plot(biased_sizes, 'r-', label='偏置引导NSGA-II', linewidth=2)
ax3.set_xlabel('代数')
ax3.set_ylabel('第一前沿大小')
ax3.set_title('收敛过程')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 偏置贡献度
ax4 = axes[1, 1]
if biased_nsga2.bias_statistics:
    bias_names = list(biased_nsga2.bias_statistics.keys())
    contributions = list(biased_nsga2.bias_statistics.values())
    
    wedges, texts, autotexts = ax4.pie(
        contributions, labels=bias_names, 
        autopct='%1.1f%%', startangle=90
    )
    ax4.set_title('偏置贡献度分布')
    
    # 美化文字
    for autotext in autotexts:
        autotext.set_color('white')
        autotext.set_fontweight('bold')

plt.tight_layout()
plt.show()

# 输出统计结果
print("\n" + "="*50)
print("优化结果统计：")
print("="*50)
print(f"\n标准NSGA-II：")
print(f"  解的数量: {len(standard_pareto)}")
print(f"  可行解比例: {standard_feasible:.2%}")
print(f"  成本范围: [{standard_objectives[:, 0].min():.3f}, {standard_objectives[:, 0].max():.3f}]")
print(f"  风险范围: [{standard_objectives[:, 1].min():.3f}, {standard_objectives[:, 1].max():.3f}]")

print(f"\n偏置引导NSGA-II：")
print(f"  解的数量: {len(biased_pareto)}")
print(f"  可行解比例: {biased_feasible:.2%}")
print(f"  成本范围: [{biased_objectives[:, 0].min():.3f}, {biased_objectives[:, 0].max():.3f}]")
print(f"  风险范围: [{biased_objectives[:, 1].min():.3f}, {biased_objectives[:, 1].max():.3f}]")

# 选择推荐投资组合
def select_investment_portfolio(pareto_front, preference_type="risk_averse"):
    """根据偏好选择投资组合"""
    if preference_type == "risk_averse":
        # 风险厌恶：选择风险最小的
        best = min(pareto_front, key=lambda x: x.objectives[1])
    elif preference_type == "return_seeking":
        # 收益导向：选择收益最高的
        best = max(pareto_front, key=lambda x: x.objectives[0])
    else:
        # 平衡型：选择最接近理想点的
        best = min(pareto_front, 
                  key=lambda x: np.sqrt((x.objectives[0]-0.2)**2 + (x.objectives[1]-0.1)**2))
    
    return best

# 选择不同偏好的投资组合
print("\n推荐投资组合：")
print("-"*30)

risk_averse = select_investment_portfolio(biased_pareto, "risk_averse")
return_seeking = select_investment_portfolio(biased_pareto, "return_seeking")
balanced = select_investment_portfolio(biased_pareto, "balanced")

print(f"\n风险厌恶型：")
print(f"  成本: {risk_averse.objectives[0]:.3f}, 风险: {risk_averse.objectives[1]:.3f}")
print(f"\n收益导向型：")
print(f"  成本: {return_seeking.objectives[0]:.3f}, 风险: {return_seeking.objectives[1]:.3f}")
print(f"\n平衡型：")
print(f"  成本: {balanced.objectives[0]:.3f}, 风险: {balanced.objectives[1]:.3f}")

## 偏置在多目标优化中的核心价值

### 1. 引导收敛
- **Pareto引导偏置**：智能识别稀疏区域，自动填补解集空白
- **偏好导向偏置**：将搜索聚焦于决策者关心的区域
- **收敛加速**：比随机搜索更快找到高质量Pareto前沿

### 2. 改善多样性
- **多样性保持偏置**：确保解集分布均匀
- **自适应调整**：根据当前解集状态动态调整策略
- **避免早熟**：防止陷入局部Pareto前沿

### 3. 处理约束
- **约束感知偏置**：自动避开不可行区域
  - 预算约束：自动控制投资规模
  - 质量约束：避免高风险配置
  - 政策约束：满足监管要求
- **可行解优先**：优先保留可行解

### 4. 融入业务知识
- **行业经验**：将专家经验编码为偏置
- **市场条件**：根据市场环境调整搜索策略
- **风险偏好**：反映投资者的风险态度
- **历史数据**：从历史投资中学习

## 实际应用建议

1. **金融投资**：
   - 必须使用约束偏置控制风险
   - 偏好偏置反映投资者类型
   - 多样性偏置避免资产过度集中

2. **工程设计**：
   - Pareto偏置改善性能权衡
   - 约束偏置确保设计可行
   - 成本偏置控制预算超支

3. **供应链优化**：
   - 多目标平衡成本、时间、质量
   - 偏置处理供应商约束
   - 鲁棒性偏置应对不确定性

4. **能源调度**：
   - 环保偏置减少碳排放
   - 成本偏置控制运营费用
   - 可靠性偏置保证稳定供应

偏置系统让NSGA-II从一个通用算法转变为一个**智能化的业务优化工具**，能够理解和执行复杂的业务规则和决策者偏好。